# 04 Comparison and Condition Data

This notebook combines the reference, RADAR, and LiDAR velocity results and prepares the condition data used by the later condition-based analyses.

The original comparison and condition-data logic from `condition_velocity.ipynb` is retained.

The final comparison and condition tables are saved to `results/condition/`.

In [1]:
import sys
sys.path.append("../..")

from pathlib import Path

import pandas as pd
from truckscenes import TruckScenes

from src.config import DATA_ROOT
from src.comparison import build_velocity_comparison_df
from src.condition.condition_config import CONDITIONS
from src.condition.condition_data import (
    build_condition_table,
    build_condition_inventory,
)

In [2]:
RESULT_DIR = Path("../../results/condition")

## 1. Load Previous Results

In [ ]:
keyframe_df = pd.read_csv(RESULT_DIR / "keyframe_df.csv")
vehicle_reference_df = pd.read_csv(RESULT_DIR / "vehicle_reference_df.csv")
radar_velocity_df = pd.read_csv(RESULT_DIR / "radar_velocity_df.csv")
lidar_velocity_df = pd.read_csv(RESULT_DIR / "lidar_velocity_df.csv")
trucksc = TruckScenes(dataroot=DATA_ROOT)

Auto-detected latest version: v1.2-mini
Loading truckscenes tables for version v1.2-mini...
11 attribute,
18 calibrated_sensor,
27 category,
20090 ego_motion_cabin,
20089 ego_motion_chassis,
20116 ego_pose,
1094 instance,
400 sample,
25750 sample_annotation,
43556 sample_data,
10 scene,
18 sensor,
4 visibility,
10 weather_annotation,
Done loading in 0.731 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


f:\CITS3200\Boeing-RADAR-Autonomous-\.venv311\Lib\site-packages\truckscenes\truckscenes.py:100: UserWarning: The visualization dependencies are not installed on your system! Run 'pip install "truckscenes-devkit[all]"'.
  warnings.warn('''The visualization dependencies are not installed on your system! '''


## 2. Build Comparison Data

In [ ]:
local_sample_tokens = set(keyframe_df[keyframe_df["Modality"].isin(["RADAR", "LiDAR"])]["Sample Token"])

local_vehicle_reference_df = vehicle_reference_df[
    vehicle_reference_df["start_sample_token"].isin(local_sample_tokens)
    & vehicle_reference_df["end_sample_token"].isin(local_sample_tokens)
].copy()

In [5]:
comparison_df = build_velocity_comparison_df(
    local_vehicle_reference_df,
    radar_velocity_df,
    lidar_velocity_df,
)

interval_key = ["start_annotation_token", "end_annotation_token"]

comparison_df = comparison_df.merge(
    radar_velocity_df[interval_key + ["is_stable_2d"]],
    on=interval_key,
    how="left",
)

comparison_df["radar_reconstructable_available"] = comparison_df["radar_velocity_available"]
comparison_df["radar_stable_available"] = comparison_df["is_stable_2d"].fillna(False)
comparison_df["radar_velocity_available"] = comparison_df["radar_stable_available"]

In [6]:
print("Reference intervals:", len(comparison_df))
print("RADAR reconstructable:", comparison_df["radar_reconstructable_available"].sum())
print("RADAR stable:", comparison_df["radar_stable_available"].sum())
print("LiDAR available:", comparison_df["lidar_velocity_available"].sum())

Reference intervals: 15743
RADAR reconstructable: 7706
RADAR stable: 3016
LiDAR available: 8007


In [ ]:
comparison_df.to_csv(RESULT_DIR / "comparison_df.csv", index=False)
print("Comparison data saved.")

Comparison data saved.


## 3. Build Condition Data

In [ ]:
local_sample_tokens = set(keyframe_df[keyframe_df["Modality"].isin(["RADAR", "LiDAR"])]["Sample Token"])

In [ ]:
condition_df = build_condition_table(trucksc, sample_tokens=local_sample_tokens)
condition_inventory = build_condition_inventory(condition_df, CONDITIONS)
condition_inventory

,condition,value,scenes,samples,objects,annotations
0,area,city,4,160,569,14663
1,area,highway,4,160,468,9165
2,area,terminal,2,80,57,1922
3,weather,clear,4,160,436,10464
4,weather,overcast,3,120,425,9109
5,weather,rain,2,80,142,3612
6,weather,snow,1,40,91,2565
7,visibility,1,10,400,798,8346
8,visibility,2,10,387,450,2176
9,visibility,3,10,376,445,2034


## 4. Save Condition Data

In [ ]:
condition_df.to_csv(RESULT_DIR / "condition_df.csv", index=False)
print("Condition data saved.")

Condition data saved.


## 5. Setup Summary

In [11]:
setup_summary = pd.DataFrame({
    "dataset": [
        "Reference intervals",
        "RADAR velocity intervals",
        "LiDAR velocity intervals",
        "Comparison intervals",
        "Condition annotations",
    ],
    "rows": [
        len(local_vehicle_reference_df),
        len(radar_velocity_df),
        len(lidar_velocity_df),
        len(comparison_df),
        len(condition_df),
    ],
})

setup_summary

,dataset,rows
0,Reference intervals,15743
1,RADAR velocity intervals,7706
2,LiDAR velocity intervals,8007
3,Comparison intervals,15743
4,Condition annotations,25750
